In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%cd /workspace/Code/experimental/

/workspace/Code/experimental


In [3]:
from data.datamodule import TextDataModule
from src.modules.core.encoder import TransformerEncoder
from src.modules.mlm_baseline.mlm_module import MLMModule
import lightning as L
import torch
from data.tokenizer import TokenizerWrapper
from lightning.pytorch.loggers import WandbLogger

In [4]:
PATH = './logs/mlm_baseline/colab/epoch=13-step=25648.ckpt'

In [5]:
dm = TextDataModule(
    train_path="data/raw/tinystories_train.parquet",
    val_path="data/raw/tinystories_val.parquet",
    tokenizer_name="bert-base-uncased",
    window_size=128,
    alpha=0.15,
    batch_size=256,
    num_workers=1,
)

In [6]:
encoder = TransformerEncoder(
    vocab_size=30522,   
    d_model=128,        # <-- Passé de 8 à 128 : donne la capacité d'apprendre la grammaire
    n_heads=4,          # <-- Passé de 1 à 4 : chaque tête gère des sous-vecteurs de dim 32
    n_layers=3,         # <-- Réduit de 4 à 3 : sur un petit d_model, 3 couches suffisent amplement
    d_ff=512,           # <-- Ajusté (4 * d_model) pour respecter les standards des Transformers
    max_seq_len=128,
)

dm = TokenizerWrapper()

model = MLMModule.load_from_checkpoint(
    PATH, 
    encoder=encoder
)

model.eval()
model.freeze()

Chargement du tokenizer depuis le cache local : ./data/tokenizers/bert-base-uncased


MLMModule(
  (encoder): TransformerEncoder(
    (token_embedding): Embedding(30522, 128, padding_idx=0)
    (position_embedding): Embedding(128, 128)
    (embed_dropout): Dropout(p=0.1, inplace=False)
    (blocks): ModuleList(
      (0-2): 3 x TransformerBlock(
        (attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (ff): Sequential(
          (0): Linear(in_features=128, out_features=512, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=512, out_features=128, bias=True)
          (3): Dropout(p=0.1, inplace=False)
        )
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )
    )
    (final_norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
  )
  (head): MLMHead(
    (norm): LayerNorm((128,), eps=1

In [7]:
# 0. Récupération du tokenizer
tokenizer = dm.tokenizer  

text = "Once upon a time, there was a little girl named [MASK]. She was [MASK] [MASK] [MASK] and she [MASK] purple things. One day, she was walking in the park when she saw a [MASK] pastry. It was covered in purple [MASK], and she wanted to try it! "
# 1. Tokenisation
inputs = tokenizer.encode(text)  

# 2. Construction de l'attention mask
attention_mask_list = [1 if x != tokenizer.pad_token_id else 0 for x in inputs]

print("inputs")
print(inputs)
print(attention_mask_list)

# 3. Trouver TOUS les indices des tokens [MASK]
mask_token_id = tokenizer.mask_token_id

# On liste toutes les positions où apparaît le token [MASK]
mask_indices = [i for i, token_id in enumerate(inputs) if token_id == mask_token_id]

if not mask_indices:
    raise ValueError(f"Le token de masquage ({tokenizer.mask_token}) est introuvable.")

# 4. Conversion en tenseurs avec dimension Batch (B=1)
input_ids = torch.tensor([inputs], dtype=torch.long, device=model.device)                  # Shape: (1, L)
attention_mask = torch.tensor([attention_mask_list], dtype=torch.long, device=model.device)  # Shape: (1, L)

# masked_positions devient de taille (1, M) où M est le nombre de masques trouvés
masked_positions = torch.tensor([mask_indices], dtype=torch.long, device=model.device)      # Shape: (1, M)

print("batch inputs id", input_ids)
print("attn mask", attention_mask)
print("masked pos", masked_positions)

# 5. Passage dans le modèle
model.eval()
with torch.no_grad():
    hidden_states = model.encoder(input_ids, attention_mask)
    
    # Ta fonction extrait les M positions valides -> gathered a une shape (M, D) car N = M ici (Batch=1)
    gathered, valid = model._gather_masked_positions(hidden_states, masked_positions, attention_mask)
    
    # La tête MLM projette vers le vocabulaire -> logits a une shape (M, vocab_size)
    logits = model.head(gathered) 
    
    # Argmax sur la dimension du vocabulaire (-1) pour obtenir les IDs des M tokens prédits
    predicted_token_ids = torch.argmax(logits, dim=-1).tolist()  # Converti en liste Python pour le décodage

# 6. Décodage et affichage du résultat
# On décode chaque token individuellement
predicted_words = [tokenizer.decode([token_id]).strip() for token_id in predicted_token_ids]

print(f"Indices des masques détectés : {mask_indices}")
print(f"Mots prédits dans l'ordre    : {predicted_words}")

# Reconstruction visuelle de la phrase
text_sections = text.split(tokenizer.mask_token)
completed_text = ""
for i, section in enumerate(text_sections[:-1]):
    completed_text += f"{section}**{predicted_words[i]}**"
completed_text += text_sections[-1]

print("\nPhrase complétée :")
print(completed_text)

inputs
[101, 2320, 2588, 1037, 2051, 1010, 2045, 2001, 1037, 2210, 2611, 2315, 103, 1012, 2016, 2001, 103, 103, 103, 1998, 2016, 103, 6379, 2477, 1012, 2028, 2154, 1010, 2016, 2001, 3788, 1999, 1996, 2380, 2043, 2016, 2387, 1037, 103, 27060, 1012, 2009, 2001, 3139, 1999, 6379, 103, 1010, 1998, 2016, 2359, 2000, 3046, 2009, 999, 102]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
batch inputs id tensor([[  101,  2320,  2588,  1037,  2051,  1010,  2045,  2001,  1037,  2210,
          2611,  2315,   103,  1012,  2016,  2001,   103,   103,   103,  1998,
          2016,   103,  6379,  2477,  1012,  2028,  2154,  1010,  2016,  2001,
          3788,  1999,  1996,  2380,  2043,  2016,  2387,  1037,   103, 27060,
          1012,  2009,  2001,  3139,  1999,  6379,   103,  1010,  1998,  2016,
          2359,  2000,  3046,  2009,   999,   102]])
attn mask tensor([[1, 1, 1, 1, 1